In [52]:
# ============================================================
# 04e1_blending_voting_ensembles.ipynb
# Purpose:
#   Build and compare ensemble models using saved base model predictions.
#
# Ensemble families:
#   EASTACK    - Equal Average
#   WASTACK    - Optimized Weighted Average
#   LRSTACK    - Logistic Regression Stack
#   RIDGESTACK - Ridge Logistic Stack
#   LASSOSTACK - Lasso Logistic Stack
#   RFSTACK    - Random Forest Meta-Learner
#   XGBSTACK   - XGBoost Meta-Learner
#   RASTACK    - Rank Average
#   BAYSTACK   - Bayesian / Performance Weighted Average
#   NNSTACK    - Neural Net Blender
# ============================================================

import os
import json
import joblib
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import softmax
from scipy.stats import ks_2samp, rankdata

from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Setup complete.")

Setup complete.


In [20]:
# ============================================================
# Helper Functions
# ============================================================

def calculate_ks(y_true, y_pred):
    """
    Calculate KS statistic between positive and negative classes.
    """
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    return ks_2samp(
        y_pred[y_true == 1],
        y_pred[y_true == 0]
    ).statistic


def safe_log_loss(y_true, y_pred):
    """
    Calculate log loss with clipped probabilities.
    """
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return log_loss(y_true, y_pred)


def evaluate_predictions(y_train, train_pred, y_val, val_pred):
    """
    Return common model evaluation metrics.
    """
    return {
        "train_auc": float(roc_auc_score(y_train, train_pred)),
        "val_auc": float(roc_auc_score(y_val, val_pred)),
        "train_ks": float(calculate_ks(y_train, train_pred)),
        "val_ks": float(calculate_ks(y_val, val_pred)),
        "train_logloss": float(safe_log_loss(y_train, train_pred)),
        "val_logloss": float(safe_log_loss(y_val, val_pred)),
    }


def get_ensemble_data(train_pred_matrix, val_pred_matrix, base_model_list):
    """
    Return train/validation prediction matrices for selected base models.
    """

    missing_train = [m for m in base_model_list if m not in train_pred_matrix.columns]
    missing_val = [m for m in base_model_list if m not in val_pred_matrix.columns]

    if missing_train:
        raise KeyError(f"Missing train prediction columns: {missing_train}")

    if missing_val:
        raise KeyError(f"Missing validation prediction columns: {missing_val}")

    X_train_ens = train_pred_matrix[base_model_list].copy()
    X_val_ens = val_pred_matrix[base_model_list].copy()

    return X_train_ens, X_val_ens


def register_ensemble(
    registry,
    ensemble_id,
    ensemble_name,
    ensemble_type,
    base_models,
    y_train_true,
    train_pred,
    y_val_true,
    val_pred,
    model_object=None,
    weights_table=None,
    notes=None
):
    """
    Register ensemble model results and metadata.
    """

    metrics = evaluate_predictions(
        y_train=y_train_true,
        train_pred=train_pred,
        y_val=y_val_true,
        val_pred=val_pred
    )

    registry[ensemble_id] = {
        "ensemble_id": ensemble_id,
        "ensemble_name": ensemble_name,
        "ensemble_type": ensemble_type,
        "base_models": list(base_models),
        **metrics,
        "model_object": model_object,
        "weights_table": weights_table,
        "train_predictions": np.asarray(train_pred),
        "val_predictions": np.asarray(val_pred),
        "created_ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "notes": notes
    }

    print(f"{ensemble_id} registered.")
    print(f"Train AUC: {metrics['train_auc']:.4f} | Val AUC: {metrics['val_auc']:.4f}")
    print(f"Train KS:  {metrics['train_ks']:.4f} | Val KS:  {metrics['val_ks']:.4f}")
    print(f"Train LL:  {metrics['train_logloss']:.4f} | Val LL:  {metrics['val_logloss']:.4f}")

    return registry

In [2]:
# ============================================================
# Paths
# ============================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)
print("Model Directory:", MODEL_DIR)
print("Config Directory:", CONFIG_DIR)

Project Root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs
Model Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/saved_models
Config Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/model_configs


In [7]:
# ============================================================
# Load Model-Ready Datasets
# ============================================================

df_tree = pd.read_parquet(DATA_DIR / "df_tree_model_ready.parquet")
df_scaled = pd.read_parquet(DATA_DIR / "df_scaled_model_ready.parquet")
df_woe = pd.read_parquet(DATA_DIR / "df_logit_woe_ready.parquet")
df_bins = pd.read_parquet(DATA_DIR / "df_logit_bins_ready.parquet")

print("Tree dataset:", df_tree.shape)
print("Scaled dataset:", df_scaled.shape)
print("WOE dataset:", df_woe.shape)
print("Binned dataset:", df_bins.shape)

display(df_tree.head())

Tree dataset: (149390, 16)
Scaled dataset: (149390, 16)
WOE dataset: (149390, 12)
Binned dataset: (149390, 12)


,SeriousDlqin2yrs,age,age_sq,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents_median,NumberOfDependents_missing_flag,MonthlyIncome_median,MonthlyIncome_missing_flag,DebtRatio_log,DebtRatio_high_flag,RevolvingUtilization_log,RevolvingUtilization_high_flag
0,1,45,2025,2,0,0,13,6,2.0,0,9120.0,0,0.589442,0,0.568789,0
1,0,40,1600,0,0,0,4,0,1.0,0,2600.0,0,0.115002,0,0.671490,0
2,0,38,1444,1,0,1,2,0,0.0,0,3042.0,0,0.081684,0,0.505721,0
3,0,30,900,0,0,0,5,0,0.0,0,3300.0,0,0.035415,0,0.210107,0
4,0,49,2401,1,0,0,7,1,0.0,0,63588.0,0,0.024620,0,0.645657,0


In [8]:
# ============================================================
# Load Saved Base Models
# ============================================================

champion_models = {
    "LGB002": joblib.load(MODEL_DIR / "LGB002.joblib"),
    "XGB004": joblib.load(MODEL_DIR / "XGB004.joblib"),
    "CAT005": joblib.load(MODEL_DIR / "CAT005.joblib"),
    "NN003":  joblib.load(MODEL_DIR / "NN003.joblib"),
    "BNB018": joblib.load(MODEL_DIR / "BNB018.joblib"),
    "WGNB012": joblib.load(MODEL_DIR / "WGNB012.joblib"),
    "M009":   joblib.load(MODEL_DIR / "M009.joblib")
}

print("Loaded models:")
print(list(champion_models.keys()))

Loaded models:
['LGB002', 'XGB004', 'CAT005', 'NN003', 'BNB018', 'WGNB012', 'M009']


In [9]:
# ============================================================
# Load Base Model Configs Robustly
# ============================================================

model_configs = {}

for model_id in champion_models.keys():

    config_path = CONFIG_DIR / f"{model_id}_config.json"

    if not config_path.exists():
        raise FileNotFoundError(f"Missing config file: {config_path}")

    with open(config_path, "r") as f:
        cfg = json.load(f)

    feature_list = cfg.get("features")

    if feature_list is None:
        feature_list = cfg.get("metadata", {}).get("features")

    if feature_list is None:
        raise KeyError(f"No feature list found in config for {model_id}")

    cfg["resolved_features"] = feature_list
    model_configs[model_id] = cfg

    print(model_id, "features:", len(feature_list))

features = {
    model_id: cfg["resolved_features"]
    for model_id, cfg in model_configs.items()
}

print("Feature dictionary created.")

LGB002 features: 15
XGB004 features: 15
CAT005 features: 15
NN003 features: 15
BNB018 features: 41
WGNB012 features: 5
M009 features: 7
Feature dictionary created.


In [12]:
# ============================================================
# Recreate Common Target Split Index
# ============================================================

target = "SeriousDlqin2yrs"

y_full = df_tree[target]

train_idx, val_idx = train_test_split(
    df_tree.index,
    test_size=0.30,
    stratify=y_full,
    random_state=42
)

y_train = df_tree.loc[train_idx, target]
y_val   = df_tree.loc[val_idx, target]

print("Train rows:", len(train_idx))
print("Validation rows:", len(val_idx))
print("Train bad rate:", y_train.mean())
print("Val bad rate  :", y_val.mean())

Train rows: 104573
Validation rows: 44817
Train bad rate: 0.06699626098514913
Val bad rate  : 0.06700582368297744


In [13]:
# ============================================================
# Build Train / Validation Feature Matrices
# ============================================================

# Tree models
X_train_tree = df_tree.loc[train_idx].drop(columns=[target])
X_val_tree   = df_tree.loc[val_idx].drop(columns=[target])

# Scaled models
X_train_scaled = df_scaled.loc[train_idx].drop(columns=[target])
X_val_scaled   = df_scaled.loc[val_idx].drop(columns=[target])

# WOE models
X_train_woe = df_woe.loc[train_idx].drop(columns=[target])
X_val_woe   = df_woe.loc[val_idx].drop(columns=[target])

# Raw binned models
X_train_bins = df_bins.loc[train_idx].drop(columns=[target])
X_val_bins   = df_bins.loc[val_idx].drop(columns=[target])

print("X_train_tree:", X_train_tree.shape)
print("X_val_tree:", X_val_tree.shape)

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)

print("X_train_woe:", X_train_woe.shape)
print("X_val_woe:", X_val_woe.shape)

print("X_train_bins:", X_train_bins.shape)
print("X_val_bins:", X_val_bins.shape)

X_train_tree: (104573, 15)
X_val_tree: (44817, 15)
X_train_scaled: (104573, 15)
X_val_scaled: (44817, 15)
X_train_woe: (104573, 11)
X_val_woe: (44817, 11)
X_train_bins: (104573, 11)
X_val_bins: (44817, 11)


In [14]:
# ============================================================
# One-Hot Encode Binned Data for BNB018
# ============================================================

X_bins_all = pd.concat(
    [X_train_bins, X_val_bins],
    axis=0
)

X_bins_ohe_all = pd.get_dummies(
    X_bins_all,
    drop_first=False,
    dtype=int
)

X_train_bins_ohe = X_bins_ohe_all.loc[train_idx].copy()
X_val_bins_ohe   = X_bins_ohe_all.loc[val_idx].copy()

print("X_train_bins_ohe:", X_train_bins_ohe.shape)
print("X_val_bins_ohe:", X_val_bins_ohe.shape)

display(X_train_bins_ohe.head())

X_train_bins_ohe: (104573, 41)
X_val_bins_ohe: (44817, 41)


,RevolvingUtilization_final_bin_=0,RevolvingUtilization_final_bin_0-5%,RevolvingUtilization_final_bin_5-15%,RevolvingUtilization_final_bin_15-30%,RevolvingUtilization_final_bin_30-50%,RevolvingUtilization_final_bin_50-75%,RevolvingUtilization_final_bin_75-100%,RevolvingUtilization_final_bin_100%+,age_final_bin_v2_21-39,age_final_bin_v2_40-49,age_final_bin_v2_50-59,age_final_bin_v2_60-69,age_final_bin_v2_70+,MonthlyIncome_final_bin_0-5000,MonthlyIncome_final_bin_5000-10000,MonthlyIncome_final_bin_10000+,MonthlyIncome_missing_flag_bin_0,MonthlyIncome_missing_flag_bin_1,DebtRatio_high_flag_bin_0,DebtRatio_high_flag_bin_1,NumberOfDependents_final_bin_0,NumberOfDependents_final_bin_1,NumberOfDependents_final_bin_2,NumberOfDependents_final_bin_3+,NumberOfDependents_missing_flag_bin_0,NumberOfDependents_missing_flag_bin_1,RealEstateLoans_final_bin_0,RealEstateLoans_final_bin_1-2,RealEstateLoans_final_bin_3+,NumberOfTime30-59DaysPastDueNotWorse_final_bin_0,NumberOfTime30-59DaysPastDueNotWorse_final_bin_1,NumberOfTime30-59DaysPastDueNotWorse_final_bin_2,NumberOfTime30-59DaysPastDueNotWorse_final_bin_3+,NumberOfTime60-89DaysPastDueNotWorse_final_bin_0,NumberOfTime60-89DaysPastDueNotWorse_final_bin_1,NumberOfTime60-89DaysPastDueNotWorse_final_bin_2,NumberOfTime60-89DaysPastDueNotWorse_final_bin_3+,NumberOfTimes90DaysLate_final_bin_0,NumberOfTimes90DaysLate_final_bin_1,NumberOfTimes90DaysLate_final_bin_2,NumberOfTimes90DaysLate_final_bin_3+
69982,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,1,0,1,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0
6532,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0
29906,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,1,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0
87276,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0
104553,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,0,1,1,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0


In [16]:
# ============================================================
# Generate Train / Validation Predictions from Base Models
# ============================================================

import statsmodels.api as sm

train_preds = pd.DataFrame(index=train_idx)
val_preds   = pd.DataFrame(index=val_idx)

for model_id, model in champion_models.items():

    feats = features[model_id]

    # Select correct source matrix
    if model_id in ["LGB002", "XGB004", "CAT005"]:
        Xtr_source = X_train_tree
        Xva_source = X_val_tree

    elif model_id == "NN003":
        Xtr_source = X_train_scaled
        Xva_source = X_val_scaled

    elif model_id == "BNB018":
        Xtr_source = X_train_bins_ohe
        Xva_source = X_val_bins_ohe

    elif model_id in ["WGNB012", "M009"]:
        Xtr_source = X_train_woe
        Xva_source = X_val_woe

    else:
        raise ValueError(f"Unknown model_id: {model_id}")

    # Validate required features
    missing_train = [f for f in feats if f not in Xtr_source.columns]
    missing_val   = [f for f in feats if f not in Xva_source.columns]

    if missing_train:
        raise KeyError(
            f"{model_id}: missing {len(missing_train)} train features. "
            f"First missing: {missing_train[:10]}"
        )

    if missing_val:
        raise KeyError(
            f"{model_id}: missing {len(missing_val)} validation features. "
            f"First missing: {missing_val[:10]}"
        )

    Xtr = Xtr_source[feats].copy()
    Xva = Xva_source[feats].copy()

    # statsmodels logistic model
    if model_id == "M009":
        Xtr = sm.add_constant(Xtr, has_constant="add")
        Xva = sm.add_constant(Xva, has_constant="add")

        Xtr = Xtr[model.model.exog_names]
        Xva = Xva[model.model.exog_names]

        train_preds[model_id] = model.predict(Xtr)
        val_preds[model_id]   = model.predict(Xva)

    # sklearn-style models
    else:
        train_preds[model_id] = model.predict_proba(Xtr)[:, 1]
        val_preds[model_id]   = model.predict_proba(Xva)[:, 1]

print("Base prediction matrices complete.")
print("Train predictions shape:", train_preds.shape)
print("Validation predictions shape:", val_preds.shape)

display(train_preds.head())
display(val_preds.head())

Base prediction matrices complete.
Train predictions shape: (104573, 7)
Validation predictions shape: (44817, 7)


,LGB002,XGB004,CAT005,NN003,BNB018,WGNB012,M009
69982,0.065840,0.048165,0.078248,0.066996,0.001184,0.000689,0.016379
6532,0.012149,0.013581,0.013283,0.009783,0.000400,0.000543,0.012556
29906,0.018572,0.019363,0.019707,0.011311,0.009517,0.000846,0.021953
87276,0.016188,0.018197,0.017387,0.017340,0.003874,0.001533,0.026137
104553,0.043454,0.041405,0.030569,0.037341,0.019541,0.003247,0.042444


,LGB002,XGB004,CAT005,NN003,BNB018,WGNB012,M009
79307,0.013140,0.010771,0.010676,0.009038,0.001793,0.000846,0.018944
49608,0.036683,0.040661,0.033510,0.036220,0.006684,0.001391,0.025318
58025,0.040508,0.045780,0.036662,0.025624,0.017279,0.004873,0.029541
105912,0.004505,0.006339,0.005610,0.005827,0.000651,0.000053,0.010068
15020,0.007284,0.007719,0.008126,0.006247,0.002485,0.000543,0.016796


In [17]:
# ============================================================
# Validate Base Predictions
# ============================================================

print("Train prediction nulls:")
display(train_preds.isna().sum())

print("Validation prediction nulls:")
display(val_preds.isna().sum())

print("Train prediction ranges:")
display(train_preds.agg(["min", "max", "mean"]).T)

print("Validation prediction ranges:")
display(val_preds.agg(["min", "max", "mean"]).T)

base_model_perf = []

for model_id in train_preds.columns:
    metrics = evaluate_predictions(
        y_train=y_train,
        train_pred=train_preds[model_id],
        y_val=y_val,
        val_pred=val_preds[model_id]
    )

    base_model_perf.append({
        "model_id": model_id,
        **metrics
    })

base_model_perf = pd.DataFrame(base_model_perf).sort_values(
    "val_auc",
    ascending=False
)

display(base_model_perf)

Train prediction nulls:


LGB002     0
XGB004     0
CAT005     0
NN003      0
BNB018     0
WGNB012    0
M009       0
dtype: int64

Validation prediction nulls:


LGB002     0
XGB004     0
CAT005     0
NN003      0
BNB018     0
WGNB012    0
M009       0
dtype: int64

Train prediction ranges:


,min,max,mean
LGB002,1.449984e-03,0.913082,0.066997
XGB004,3.708261e-03,0.857327,0.066968
CAT005,3.477911e-03,0.943326,0.066936
NN003,1.966204e-11,0.921519,0.062023
BNB018,2.856389e-05,0.999999,0.106697
WGNB012,5.336301e-05,1.000000,0.107567
M009,8.152524e-03,0.937846,0.066996


Validation prediction ranges:


,min,max,mean
LGB002,0.001267,0.903564,0.067221
XGB004,0.003575,0.829887,0.067134
CAT005,0.003568,0.905041,0.067122
NN003,0.000022,0.914467,0.062030
BNB018,0.000029,0.999999,0.106613
WGNB012,0.000053,1.000000,0.107732
M009,0.008153,0.940365,0.067172


,model_id,train_auc,val_auc,train_ks,val_ks,train_logloss,val_logloss
0,LGB002,0.874775,0.868180,0.590303,0.585756,0.173309,0.176125
2,CAT005,0.870033,0.868081,0.578907,0.586881,0.174039,0.176271
1,XGB004,0.868127,0.867015,0.574794,0.582425,0.176800,0.177337
3,NN003,0.865068,0.863171,0.571154,0.571177,0.178219,0.178940
6,M009,0.852645,0.857289,0.547688,0.559679,0.184640,0.182459
4,BNB018,0.849884,0.855575,0.543249,0.558270,0.347936,0.342268
5,WGNB012,0.851010,0.854839,0.551678,0.568055,0.671458,0.675897


In [19]:
# ============================================================
# Define Ensemble Model Set
# ============================================================

base_models_full = list(train_preds.columns)

print("Full ensemble base models:")
print(base_models_full)

Full ensemble base models:
['LGB002', 'XGB004', 'CAT005', 'NN003', 'BNB018', 'WGNB012', 'M009']


In [21]:
ensemble_registry = {}

In [22]:
# ============================================================
# EASTACK001 - Equal Average Ensemble
# ============================================================

ensemble_id = "EASTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

train_pred = X_train_ens.mean(axis=1)
val_pred   = X_val_ens.mean(axis=1)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": np.repeat(1 / len(model_list), len(model_list))
})

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Equal Average Ensemble",
    ensemble_type="Equal Probability Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Simple average of all base model predicted probabilities."
)

EASTACK001 registered.
Train AUC: 0.8676 | Val AUC: 0.8669
Train KS:  0.5754 | Val KS:  0.5858
Train LL:  0.1817 | Val LL:  0.1818


In [23]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
0,EASTACK001,0.866949,0.585805,0.181768


In [24]:
# ============================================================
# WASTACK001 - Optimized Weighted Average Ensemble
# ============================================================

ensemble_id = "WASTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

X_train_np = X_train_ens.values
X_val_np   = X_val_ens.values


# ------------------------------------------------------------
# Objective Function:
# Minimize train log loss using constrained positive weights
# ------------------------------------------------------------

def weighted_avg_logloss(raw_weights):

    weights = softmax(raw_weights)   # forces positive weights summing to 1

    pred = np.dot(X_train_np, weights)
    pred = np.clip(pred, 1e-6, 1 - 1e-6)

    return log_loss(y_train, pred)


# initialize equal starting weights
init_weights = np.zeros(len(model_list))

# optimize
result = minimize(
    weighted_avg_logloss,
    init_weights,
    method="BFGS"
)

# final weights
opt_weights = softmax(result.x)

# predictions
train_pred = np.dot(X_train_np, opt_weights)
val_pred   = np.dot(X_val_np, opt_weights)

# weights table
weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": opt_weights
}).sort_values("weight", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Optimized Weighted Average Ensemble",
    ensemble_type="Softmax-Constrained Weighted Probability Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=result,
    weights_table=weights_table,
    notes="Weights optimized on train log loss using positive weights summing to 1."
)

,model_id,weight
0,LGB002,7.847622e-01
2,CAT005,2.152157e-01
1,XGB004,1.243292e-05
3,NN003,9.719015e-06
6,M009,2.882924e-10
4,BNB018,5.499373e-53
5,WGNB012,1.830929e-65


WASTACK001 registered.
Train AUC: 0.8744 | Val AUC: 0.8685
Train KS:  0.5891 | Val KS:  0.5859
Train LL:  0.1732 | Val LL:  0.1760


In [25]:
# ============================================================
# LRSTACK001 - Logistic Regression Stacking Ensemble
# ============================================================

ensemble_id = "LRSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# train logistic regression meta-model
lr_stack = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

lr_stack.fit(X_train_ens, y_train)

# predictions
train_pred = lr_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = lr_stack.predict_proba(X_val_ens)[:, 1]

# coefficients / weights
weights_table = pd.DataFrame({
    "model_id": model_list,
    "coefficient": lr_stack.coef_[0]
}).sort_values("coefficient", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Logistic Regression Stacking Ensemble",
    ensemble_type="Logistic Regression Meta-Learner",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=lr_stack,
    weights_table=weights_table,
    notes="Plain logistic regression stack using all base model predicted probabilities."
)

,model_id,coefficient
2,CAT005,8.090764
0,LGB002,6.069858
4,BNB018,0.571122
5,WGNB012,-0.086629
1,XGB004,-0.953797
3,NN003,-1.926962
6,M009,-4.323760


LRSTACK001 registered.
Train AUC: 0.8742 | Val AUC: 0.8658
Train KS:  0.5868 | Val KS:  0.5863
Train LL:  0.1805 | Val LL:  0.1873


In [26]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
0,EASTACK001,0.866949,0.585805,0.181768
2,LRSTACK001,0.865814,0.586328,0.187264


In [27]:
# ============================================================
# RIDGESTACK001 - Ridge Logistic Stacking Ensemble
# ============================================================

ensemble_id = "RIDGESTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# train ridge logistic stack
ridge_stack = LogisticRegression(
    penalty="l2",
    C=0.50,                 # regularization strength
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

ridge_stack.fit(X_train_ens, y_train)

# predictions
train_pred = ridge_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = ridge_stack.predict_proba(X_val_ens)[:, 1]

# coefficients
weights_table = pd.DataFrame({
    "model_id": model_list,
    "coefficient": ridge_stack.coef_[0]
}).sort_values("coefficient", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Ridge Logistic Stacking Ensemble",
    ensemble_type="L2-Regularized Logistic Meta-Learner",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=ridge_stack,
    weights_table=weights_table,
    notes="Ridge logistic stack using L2 regularization on all base model probabilities."
)

,model_id,coefficient
2,CAT005,7.153058
0,LGB002,5.484275
4,BNB018,0.546718
5,WGNB012,-0.030811
1,XGB004,-0.505291
3,NN003,-1.294152
6,M009,-3.946784


RIDGESTACK001 registered.
Train AUC: 0.8742 | Val AUC: 0.8666
Train KS:  0.5865 | Val KS:  0.5870
Train LL:  0.1807 | Val LL:  0.1868


In [28]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
0,EASTACK001,0.866949,0.585805,0.181768
3,RIDGESTACK001,0.866623,0.587016,0.186805
2,LRSTACK001,0.865814,0.586328,0.187264


In [29]:
# ============================================================
# LASSOSTACK001 - Lasso Logistic Stacking Ensemble
# ============================================================

ensemble_id = "LASSOSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# train lasso logistic stack
lasso_stack = LogisticRegression(
    penalty="l1",
    C=0.50,                 # regularization strength
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

lasso_stack.fit(X_train_ens, y_train)

# predictions
train_pred = lasso_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = lasso_stack.predict_proba(X_val_ens)[:, 1]

# coefficients
weights_table = pd.DataFrame({
    "model_id": model_list,
    "coefficient": lasso_stack.coef_[0]
}).sort_values("coefficient", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Lasso Logistic Stacking Ensemble",
    ensemble_type="L1-Regularized Logistic Meta-Learner",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=lasso_stack,
    weights_table=weights_table,
    notes="Lasso logistic stack using L1 regularization. Can shrink weak model coefficients to zero."
)

,model_id,coefficient
2,CAT005,9.667501
0,LGB002,6.212238
4,BNB018,0.558552
5,WGNB012,-0.103668
3,NN003,-1.851964
1,XGB004,-2.515562
6,M009,-4.448893


LASSOSTACK001 registered.
Train AUC: 0.8738 | Val AUC: 0.8647
Train KS:  0.5851 | Val KS:  0.5856
Train LL:  0.1804 | Val LL:  0.1877


In [30]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
0,EASTACK001,0.866949,0.585805,0.181768
3,RIDGESTACK001,0.866623,0.587016,0.186805
2,LRSTACK001,0.865814,0.586328,0.187264
4,LASSOSTACK001,0.864675,0.585563,0.187735


In [31]:
# ============================================================
# RFSTACK001 - Random Forest Meta-Learner Ensemble
# ============================================================

ensemble_id = "RFSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# train random forest meta-model
rf_stack = RandomForestClassifier(
    n_estimators=300,
    max_depth=3,
    min_samples_leaf=500,
    random_state=42,
    n_jobs=-1
)

rf_stack.fit(X_train_ens, y_train)

# predictions
train_pred = rf_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = rf_stack.predict_proba(X_val_ens)[:, 1]

# feature importance
weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance": rf_stack.feature_importances_
}).sort_values("importance", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Random Forest Meta-Learner Ensemble",
    ensemble_type="Random Forest Blender",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=rf_stack,
    weights_table=weights_table,
    notes="Random Forest meta-learner using base model probabilities as features."
)

,model_id,importance
0,LGB002,0.330099
2,CAT005,0.244342
1,XGB004,0.193109
3,NN003,0.122979
6,M009,0.066591
4,BNB018,0.032982
5,WGNB012,0.009898


RFSTACK001 registered.
Train AUC: 0.8704 | Val AUC: 0.8660
Train KS:  0.5892 | Val KS:  0.5849
Train LL:  0.1743 | Val LL:  0.1767


In [32]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
0,EASTACK001,0.866949,0.585805,0.181768
3,RIDGESTACK001,0.866623,0.587016,0.186805
5,RFSTACK001,0.865967,0.584903,0.176657
2,LRSTACK001,0.865814,0.586328,0.187264
4,LASSOSTACK001,0.864675,0.585563,0.187735


In [34]:
# ============================================================
# XGBSTACK001 - XGBoost Meta-Learner Ensemble
# ============================================================

ensemble_id = "XGBSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# train xgboost meta-model
xgb_stack = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.80,
    colsample_bytree=0.80,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_stack.fit(X_train_ens, y_train)

# predictions
train_pred = xgb_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = xgb_stack.predict_proba(X_val_ens)[:, 1]

# feature importance
weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance": xgb_stack.feature_importances_
}).sort_values("importance", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="XGBoost Meta-Learner Ensemble",
    ensemble_type="XGBoost Blender",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=xgb_stack,
    weights_table=weights_table,
    notes="XGBoost meta-learner using base model probabilities as features."
)

,model_id,importance
0,LGB002,0.344381
2,CAT005,0.259767
1,XGB004,0.242071
3,NN003,0.128500
6,M009,0.011673
4,BNB018,0.006868
5,WGNB012,0.006740


XGBSTACK001 registered.
Train AUC: 0.8763 | Val AUC: 0.8678
Train KS:  0.5908 | Val KS:  0.5846
Train LL:  0.1714 | Val LL:  0.1765


In [35]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
6,XGBSTACK001,0.867825,0.584639,0.176503
0,EASTACK001,0.866949,0.585805,0.181768
3,RIDGESTACK001,0.866623,0.587016,0.186805
5,RFSTACK001,0.865967,0.584903,0.176657
2,LRSTACK001,0.865814,0.586328,0.187264
4,LASSOSTACK001,0.864675,0.585563,0.187735


In [36]:
# ============================================================
# RASTACK001 - Rank Average Ensemble
# ============================================================

ensemble_id = "RASTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# ------------------------------------------------------------
# Convert probabilities to ranks row-wise per model
# Then average normalized ranks
# ------------------------------------------------------------

train_rank_df = pd.DataFrame(index=X_train_ens.index)
val_rank_df   = pd.DataFrame(index=X_val_ens.index)

for col in model_list:
    train_rank_df[col] = rankdata(X_train_ens[col], method="average")
    val_rank_df[col]   = rankdata(X_val_ens[col], method="average")

# normalize ranks to 0-1
train_rank_df = train_rank_df / len(train_rank_df)
val_rank_df   = val_rank_df / len(val_rank_df)

# equal average of normalized ranks
train_pred = train_rank_df.mean(axis=1)
val_pred   = val_rank_df.mean(axis=1)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": np.repeat(1 / len(model_list), len(model_list))
})

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Rank Average Ensemble",
    ensemble_type="Average of Model Prediction Ranks",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Ranks each model prediction, normalizes ranks, then averages ranks."
)

RASTACK001 registered.
Train AUC: 0.8665 | Val AUC: 0.8667
Train KS:  0.5718 | Val KS:  0.5811
Train LL:  0.8234 | Val LL:  0.8237


In [37]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
6,XGBSTACK001,0.867825,0.584639,0.176503
0,EASTACK001,0.866949,0.585805,0.181768
7,RASTACK001,0.866676,0.581106,0.823717
3,RIDGESTACK001,0.866623,0.587016,0.186805
5,RFSTACK001,0.865967,0.584903,0.176657
2,LRSTACK001,0.865814,0.586328,0.187264
4,LASSOSTACK001,0.864675,0.585563,0.187735


In [38]:
# ============================================================
# BAYSTACK001 - Bayesian / Performance Weighted Ensemble
# ============================================================

ensemble_id = "BAYSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# ------------------------------------------------------------
# Use inverse train log loss as evidence weights
# Lower log loss = stronger model
# ------------------------------------------------------------

model_scores = []

for model_id in model_list:

    ll = safe_log_loss(y_train, X_train_ens[model_id])

    score = 1 / ll

    model_scores.append({
        "model_id": model_id,
        "train_logloss": ll,
        "raw_score": score
    })

weights_table = pd.DataFrame(model_scores)

# normalize to sum to 1
weights_table["weight"] = (
    weights_table["raw_score"] /
    weights_table["raw_score"].sum()
)

weights_table = weights_table.sort_values(
    "weight",
    ascending=False
)

display(weights_table)

# ------------------------------------------------------------
# Weighted probability blend
# ------------------------------------------------------------

weights = weights_table.set_index("model_id")["weight"]

train_pred = np.zeros(len(X_train_ens))
val_pred   = np.zeros(len(X_val_ens))

for model_id in model_list:
    train_pred += X_train_ens[model_id].values * weights[model_id]
    val_pred   += X_val_ens[model_id].values * weights[model_id]

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Bayesian / Performance Weighted Ensemble",
    ensemble_type="Inverse LogLoss Weighted Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Weights based on inverse train log loss, normalized to sum to 1."
)

,model_id,train_logloss,raw_score,weight
0,LGB002,0.173309,5.770054,0.177200
2,CAT005,0.174039,5.745828,0.176456
1,XGB004,0.176800,5.656113,0.173701
3,NN003,0.178219,5.611087,0.172318
6,M009,0.184640,5.415954,0.166325
4,BNB018,0.347936,2.874095,0.088264
5,WGNB012,0.671458,1.489296,0.045737


BAYSTACK001 registered.
Train AUC: 0.8691 | Val AUC: 0.8679
Train KS:  0.5778 | Val KS:  0.5866
Train LL:  0.1767 | Val LL:  0.1772


In [39]:
pd.DataFrame([
    {
        "ensemble_id": k,
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

,ensemble_id,val_auc,val_ks,val_logloss
1,WASTACK001,0.868521,0.585882,0.175971
8,BAYSTACK001,0.867887,0.586589,0.177222
6,XGBSTACK001,0.867825,0.584639,0.176503
0,EASTACK001,0.866949,0.585805,0.181768
7,RASTACK001,0.866676,0.581106,0.823717
3,RIDGESTACK001,0.866623,0.587016,0.186805
5,RFSTACK001,0.865967,0.584903,0.176657
2,LRSTACK001,0.865814,0.586328,0.187264
4,LASSOSTACK001,0.864675,0.585563,0.187735


In [40]:
# ============================================================
# NNSTACK001 - Neural Network Blender Ensemble
# ============================================================

ensemble_id = "NNSTACK001"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

# ------------------------------------------------------------
# Neural net meta-learner
# Small network to avoid overfitting
# ------------------------------------------------------------

nn_stack = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42
)

nn_stack.fit(X_train_ens, y_train)

# predictions
train_pred = nn_stack.predict_proba(X_train_ens)[:, 1]
val_pred   = nn_stack.predict_proba(X_val_ens)[:, 1]

# no classic coefficients; use first-layer absolute weights proxy
importance = np.abs(nn_stack.coefs_[0]).mean(axis=1)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance_proxy": importance
}).sort_values("importance_proxy", ascending=False)

display(weights_table)

# register
ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Neural Network Blender Ensemble",
    ensemble_type="MLP Meta-Learner",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=nn_stack,
    weights_table=weights_table,
    notes="Two-layer neural network blender using base model probabilities as inputs."
)

,model_id,importance_proxy
2,CAT005,0.352473
0,LGB002,0.310620
6,M009,0.236836
1,XGB004,0.178433
3,NN003,0.172828
4,BNB018,0.171467
5,WGNB012,0.112177


NNSTACK001 registered.
Train AUC: 0.8745 | Val AUC: 0.8648
Train KS:  0.5879 | Val KS:  0.5861
Train LL:  0.1812 | Val LL:  0.1880


In [41]:
ensemble_summary = pd.DataFrame([
    {
        "ensemble_id": k,
        "ensemble_name": v["ensemble_name"],
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"]
    }
    for k, v in ensemble_registry.items()
]).sort_values("val_auc", ascending=False)

display(ensemble_summary)

,ensemble_id,ensemble_name,val_auc,val_ks,val_logloss
1,WASTACK001,Optimized Weighted Average Ensemble,0.868521,0.585882,0.175971
8,BAYSTACK001,Bayesian / Performance Weighted Ensemble,0.867887,0.586589,0.177222
6,XGBSTACK001,XGBoost Meta-Learner Ensemble,0.867825,0.584639,0.176503
0,EASTACK001,Equal Average Ensemble,0.866949,0.585805,0.181768
7,RASTACK001,Rank Average Ensemble,0.866676,0.581106,0.823717
3,RIDGESTACK001,Ridge Logistic Stacking Ensemble,0.866623,0.587016,0.186805
5,RFSTACK001,Random Forest Meta-Learner Ensemble,0.865967,0.584903,0.176657
2,LRSTACK001,Logistic Regression Stacking Ensemble,0.865814,0.586328,0.187264
9,NNSTACK001,Neural Network Blender Ensemble,0.864773,0.586089,0.187964
4,LASSOSTACK001,Lasso Logistic Stacking Ensemble,0.864675,0.585563,0.187735


##### Only picking the top 4 best performing ensemble models and optimizing them in the second round

In [42]:
# ============================================================
# WASTACK002 - Optimized Weighted Average (Enhanced Search)
# ============================================================

ensemble_id = "WASTACK002"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

X_train_np = X_train_ens.values
X_val_np   = X_val_ens.values

# ------------------------------------------------------------
# Objective: maximize train AUC
# minimize negative AUC
# ------------------------------------------------------------

def objective_auc(raw_weights):

    weights = softmax(raw_weights)

    pred = np.dot(X_train_np, weights)

    auc = roc_auc_score(y_train, pred)

    return -auc

best_result = None
best_score = 999

# multiple random starts
for seed in range(15):

    np.random.seed(seed)
    init = np.random.normal(0, 0.5, len(model_list))

    result = minimize(
        objective_auc,
        init,
        method="BFGS",
        options={"maxiter": 500}
    )

    if result.fun < best_score:
        best_score = result.fun
        best_result = result

# final weights
opt_weights = softmax(best_result.x)

train_pred = np.dot(X_train_np, opt_weights)
val_pred   = np.dot(X_val_np, opt_weights)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": opt_weights
}).sort_values("weight", ascending=False)

display(weights_table)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Optimized Weighted Average Ensemble v2",
    ensemble_type="Multi-start AUC Optimized Weighted Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=best_result,
    weights_table=weights_table,
    notes="Multi-start softmax weight optimization maximizing train AUC."
)

,model_id,weight
0,LGB002,0.302430
2,CAT005,0.151892
5,WGNB012,0.149683
1,XGB004,0.144861
3,NN003,0.134291
6,M009,0.065733
4,BNB018,0.051112


WASTACK002 registered.
Train AUC: 0.8701 | Val AUC: 0.8676
Train KS:  0.5805 | Val KS:  0.5877
Train LL:  0.1783 | Val LL:  0.1793


In [46]:
# ============================================================
# WASTACK003 - Multi-Start LogLoss Optimized Weighted Average
# ============================================================

ensemble_id = "WASTACK003"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

X_train_np = X_train_ens.values
X_val_np   = X_val_ens.values

def objective_logloss(raw_weights):

    weights = softmax(raw_weights)

    pred = np.dot(X_train_np, weights)
    pred = np.clip(pred, 1e-6, 1 - 1e-6)

    return log_loss(y_train, pred)

best_result = None
best_score = 999

for seed in range(20):

    np.random.seed(seed)
    init = np.random.normal(0, 0.5, len(model_list))

    result = minimize(
        objective_logloss,
        init,
        method="BFGS",
        options={"maxiter": 500}
    )

    if result.fun < best_score:
        best_score = result.fun
        best_result = result

opt_weights = softmax(best_result.x)

train_pred = np.dot(X_train_np, opt_weights)
val_pred   = np.dot(X_val_np, opt_weights)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": opt_weights
}).sort_values("weight", ascending=False)

display(weights_table)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Weighted Average Ensemble (Multi-Start LogLoss Optimized)",
    ensemble_type="Softmax Multi-Start LogLoss Weighted Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=best_result,
    weights_table=weights_table,
    notes="Multiple random starts optimizing train log loss."
)

,model_id,weight
0,LGB002,7.840601e-01
2,CAT005,2.139814e-01
5,WGNB012,1.829859e-03
4,BNB018,1.190700e-04
1,XGB004,9.526001e-06
6,M009,3.752868e-08
3,NN003,8.465986e-12


WASTACK003 registered.
Train AUC: 0.8744 | Val AUC: 0.8685
Train KS:  0.5889 | Val KS:  0.5861
Train LL:  0.1732 | Val LL:  0.1760


In [47]:
# ============================================================
# WASTACK004 - Multi-Start Brier Optimized Weighted Average
# ============================================================

ensemble_id = "WASTACK004"

def objective_brier(raw_weights):

    weights = softmax(raw_weights)

    pred = np.dot(X_train_np, weights)
    pred = np.clip(pred, 1e-6, 1 - 1e-6)

    return brier_score_loss(y_train, pred)

best_result = None
best_score = 999

for seed in range(20):

    np.random.seed(seed)
    init = np.random.normal(0, 0.5, len(model_list))

    result = minimize(
        objective_brier,
        init,
        method="BFGS",
        options={"maxiter": 500}
    )

    if result.fun < best_score:
        best_score = result.fun
        best_result = result

opt_weights = softmax(best_result.x)

train_pred = np.dot(X_train_np, opt_weights)
val_pred   = np.dot(X_val_np, opt_weights)

weights_table = pd.DataFrame({
    "model_id": model_list,
    "weight": opt_weights
}).sort_values("weight", ascending=False)

display(weights_table)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Weighted Average Ensemble (Multi-Start Brier Optimized)",
    ensemble_type="Softmax Multi-Start Brier Weighted Average",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=best_result,
    weights_table=weights_table,
    notes="Multiple random starts optimizing train Brier score."
)

,model_id,weight
2,CAT005,8.506766e-01
0,LGB002,1.493234e-01
6,M009,1.216486e-15
1,XGB004,1.275075e-19
3,NN003,2.386152e-25
5,WGNB012,5.474581e-67
4,BNB018,5.115286e-122


WASTACK004 registered.
Train AUC: 0.8712 | Val AUC: 0.8684
Train KS:  0.5808 | Val KS:  0.5879
Train LL:  0.1738 | Val LL:  0.1761


In [49]:
# ============================================================
# WASTACK005 - Reduced Sparse Weighted Average Ensemble
# Based on WASTACK003 learned weights
# ============================================================

# ------------------------------------------------------------
# Inspect WASTACK003 weights first
# ------------------------------------------------------------

w003_weights = ensemble_registry["WASTACK003"]["weights_table"].copy()

display(w003_weights)

# ------------------------------------------------------------
# Keep only meaningful weights >= 1%
# (adjust threshold if desired)
# ------------------------------------------------------------

reduced_models = (
    w003_weights.loc[w003_weights["weight"] >= 0.01, "model_id"]
    .tolist()
)

print("Selected models for WASTACK005:")
print(reduced_models)

# ============================================================
# Build WASTACK005
# ============================================================

ensemble_id = "WASTACK005"

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=reduced_models
)

X_train_np = X_train_ens.values
X_val_np   = X_val_ens.values

# ------------------------------------------------------------
# Optimize logloss again on reduced set
# ------------------------------------------------------------

def objective_logloss(raw_weights):

    weights = softmax(raw_weights)

    pred = np.dot(X_train_np, weights)
    pred = np.clip(pred, 1e-6, 1 - 1e-6)

    return log_loss(y_train, pred)

best_result = None
best_score = 999

for seed in range(20):

    np.random.seed(seed)
    init = np.random.normal(0, 0.5, len(reduced_models))

    result = minimize(
        objective_logloss,
        init,
        method="BFGS",
        options={"maxiter": 500}
    )

    if result.fun < best_score:
        best_score = result.fun
        best_result = result

opt_weights = softmax(best_result.x)

train_pred = np.dot(X_train_np, opt_weights)
val_pred   = np.dot(X_val_np, opt_weights)

weights_table = pd.DataFrame({
    "model_id": reduced_models,
    "weight": opt_weights
}).sort_values("weight", ascending=False)

display(weights_table)

# ------------------------------------------------------------
# Register
# ------------------------------------------------------------

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Reduced Sparse Weighted Average Ensemble",
    ensemble_type="Reduced Softmax LogLoss Weighted Average",
    base_models=reduced_models,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=best_result,
    weights_table=weights_table,
    notes="Reduced weighted ensemble keeping only models with >=1% WASTACK003 weight."
)

,model_id,weight
0,LGB002,7.840601e-01
2,CAT005,2.139814e-01
5,WGNB012,1.829859e-03
4,BNB018,1.190700e-04
1,XGB004,9.526001e-06
6,M009,3.752868e-08
3,NN003,8.465986e-12


Selected models for WASTACK005:
['LGB002', 'CAT005']


,model_id,weight
0,LGB002,0.783974
1,CAT005,0.216026


WASTACK005 registered.
Train AUC: 0.8744 | Val AUC: 0.8685
Train KS:  0.5891 | Val KS:  0.5858
Train LL:  0.1732 | Val LL:  0.1760


In [53]:
# ============================================================
# XGBSTACK002 - Tuned XGBoost Meta-Learner
# ============================================================

ensemble_id = "XGBSTACK002"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

xgb_stack_002 = XGBClassifier(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.02,
    min_child_weight=50,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.1,
    reg_lambda=5.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_stack_002.fit(X_train_ens, y_train)

train_pred = xgb_stack_002.predict_proba(X_train_ens)[:, 1]
val_pred   = xgb_stack_002.predict_proba(X_val_ens)[:, 1]

weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance": xgb_stack_002.feature_importances_
}).sort_values("importance", ascending=False)

display(weights_table)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Tuned XGBoost Meta-Learner Ensemble",
    ensemble_type="Regularized XGBoost Blender",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=xgb_stack_002,
    weights_table=weights_table,
    notes="Tuned XGBoost meta-learner with shallow trees and stronger regularization."
)

,model_id,importance
0,LGB002,0.504953
2,CAT005,0.236144
3,NN003,0.132203
1,XGB004,0.090794
6,M009,0.013406
5,WGNB012,0.012317
4,BNB018,0.010182


XGBSTACK002 registered.
Train AUC: 0.8770 | Val AUC: 0.8675
Train KS:  0.5916 | Val KS:  0.5831
Train LL:  0.1713 | Val LL:  0.1765


In [55]:
# ============================================================
# XGBSTACK003 - Moderate Depth Challenger
# ============================================================

ensemble_id = "XGBSTACK003"

model_list = base_models_full

X_train_ens, X_val_ens = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=model_list
)

xgb_stack_003 = XGBClassifier(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=3.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_stack_003.fit(X_train_ens, y_train)

train_pred = xgb_stack_003.predict_proba(X_train_ens)[:,1]
val_pred   = xgb_stack_003.predict_proba(X_val_ens)[:,1]

weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance": xgb_stack_003.feature_importances_
}).sort_values("importance", ascending=False)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="XGB Moderate Depth Ensemble",
    ensemble_type="XGBoost Blender",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=xgb_stack_003,
    weights_table=weights_table,
    notes="Moderate depth XGB challenger."
)

XGBSTACK003 registered.
Train AUC: 0.8794 | Val AUC: 0.8668
Train KS:  0.5954 | Val KS:  0.5846
Train LL:  0.1699 | Val LL:  0.1771


In [57]:
# ============================================================
# XGBSTACK004 - Conservative Many Trees
# ============================================================

ensemble_id = "XGBSTACK004"

xgb_stack_004 = XGBClassifier(
    n_estimators=500,
    max_depth=2,
    learning_rate=0.01,
    min_child_weight=50,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.10,
    reg_lambda=8.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_stack_004.fit(X_train_ens, y_train)

train_pred = xgb_stack_004.predict_proba(X_train_ens)[:,1]
val_pred   = xgb_stack_004.predict_proba(X_val_ens)[:,1]

weights_table = pd.DataFrame({
    "model_id": model_list,
    "importance": xgb_stack_004.feature_importances_
}).sort_values("importance", ascending=False)

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="XGB Conservative Ensemble",
    ensemble_type="XGBoost Blender",
    base_models=model_list,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=xgb_stack_004,
    weights_table=weights_table,
    notes="Low learning rate conservative XGB."
)

XGBSTACK004 registered.
Train AUC: 0.8764 | Val AUC: 0.8678
Train KS:  0.5911 | Val KS:  0.5831
Train LL:  0.1716 | Val LL:  0.1764


In [59]:
# ============================================================
# BAYSTACK002 - Sparse Top Model Bayesian Ensemble
# Keep strongest evidence models only
# ============================================================

ensemble_id = "BAYSTACK002"

top_models = ["LGB002", "CAT005", "XGB004"]

X_train_top, X_val_top = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=top_models
)

scores = []

for m in top_models:
    ll = safe_log_loss(y_train, X_train_top[m])
    score = 1 / ll
    scores.append((m, score))

weights_table = pd.DataFrame(scores, columns=["model_id", "raw_score"])
weights_table["weight"] = (
    weights_table["raw_score"] /
    weights_table["raw_score"].sum()
)

weights_table = weights_table.sort_values("weight", ascending=False)

display(weights_table)

weights = weights_table.set_index("model_id")["weight"]

train_pred = np.zeros(len(X_train_top))
val_pred   = np.zeros(len(X_val_top))

for m in top_models:
    train_pred += X_train_top[m].values * weights[m]
    val_pred   += X_val_top[m].values * weights[m]

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Sparse Bayesian Ensemble",
    ensemble_type="Inverse LogLoss Top Model Blend",
    base_models=top_models,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Top model inverse-logloss Bayesian weighting."
)

,model_id,raw_score,weight
0,LGB002,5.770054,0.336015
1,CAT005,5.745828,0.334605
2,XGB004,5.656113,0.329380


BAYSTACK002 registered.
Train AUC: 0.8724 | Val AUC: 0.8687
Train KS:  0.5838 | Val KS:  0.5876
Train LL:  0.1741 | Val LL:  0.1761


In [60]:
# ============================================================
# BAYSTACK003 - Hybrid AUC + LogLoss Bayesian Ensemble
# ============================================================

ensemble_id = "BAYSTACK003"

scores = []

for m in base_models_full:

    auc = roc_auc_score(y_train, train_preds[m])
    ll  = safe_log_loss(y_train, train_preds[m])

    score = auc / ll

    scores.append((m, auc, ll, score))

weights_table = pd.DataFrame(
    scores,
    columns=["model_id", "train_auc", "train_logloss", "raw_score"]
)

weights_table["weight"] = (
    weights_table["raw_score"] /
    weights_table["raw_score"].sum()
)

weights_table = weights_table.sort_values("weight", ascending=False)

display(weights_table)

weights = weights_table.set_index("model_id")["weight"]

train_pred = np.zeros(len(train_preds))
val_pred   = np.zeros(len(val_preds))

for m in base_models_full:
    train_pred += train_preds[m].values * weights[m]
    val_pred   += val_preds[m].values * weights[m]

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Hybrid Bayesian Ensemble",
    ensemble_type="AUC / LogLoss Evidence Weighted Blend",
    base_models=base_models_full,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Weights proportional to AUC divided by logloss."
)

,model_id,train_auc,train_logloss,raw_score,weight
0,LGB002,0.874775,0.173309,5.047500,0.179379
2,CAT005,0.870033,0.174039,4.999058,0.177658
1,XGB004,0.868127,0.176800,4.910226,0.174501
3,NN003,0.865068,0.178219,4.853970,0.172502
6,M009,0.852645,0.184640,4.617885,0.164112
4,BNB018,0.849884,0.347936,2.442646,0.086807
5,WGNB012,0.851010,0.671458,1.267406,0.045041


BAYSTACK003 registered.
Train AUC: 0.8692 | Val AUC: 0.8679
Train KS:  0.5779 | Val KS:  0.5863
Train LL:  0.1767 | Val LL:  0.1772


In [61]:
# ============================================================
# BAYSTACK004 - Top 2 Bayesian Ensemble
# ============================================================

ensemble_id = "BAYSTACK004"

top2_models = ["LGB002", "CAT005"]

X_train_top2, X_val_top2 = get_ensemble_data(
    train_pred_matrix=train_preds,
    val_pred_matrix=val_preds,
    base_model_list=top2_models
)

scores = []

for m in top2_models:
    ll = safe_log_loss(y_train, X_train_top2[m])
    score = 1 / ll
    scores.append((m, score))

weights_table = pd.DataFrame(scores, columns=["model_id", "raw_score"])
weights_table["weight"] = (
    weights_table["raw_score"] /
    weights_table["raw_score"].sum()
)

weights_table = weights_table.sort_values("weight", ascending=False)

display(weights_table)

weights = weights_table.set_index("model_id")["weight"]

train_pred = np.zeros(len(X_train_top2))
val_pred   = np.zeros(len(X_val_top2))

for m in top2_models:
    train_pred += X_train_top2[m].values * weights[m]
    val_pred   += X_val_top2[m].values * weights[m]

ensemble_registry = register_ensemble(
    registry=ensemble_registry,
    ensemble_id=ensemble_id,
    ensemble_name="Top 2 Bayesian Ensemble",
    ensemble_type="Inverse LogLoss Two Model Blend",
    base_models=top2_models,
    y_train_true=y_train,
    train_pred=train_pred,
    y_val_true=y_val,
    val_pred=val_pred,
    model_object=None,
    weights_table=weights_table,
    notes="Two strongest model Bayesian blend."
)

,model_id,raw_score,weight
0,LGB002,5.770054,0.501052
1,CAT005,5.745828,0.498948


BAYSTACK004 registered.
Train AUC: 0.8733 | Val AUC: 0.8687
Train KS:  0.5844 | Val KS:  0.5875
Train LL:  0.1733 | Val LL:  0.1759


In [62]:
# ============================================================
# Ensemble Leaderboard / Ranking Table
# ============================================================

ensemble_summary = pd.DataFrame([
    {
        "ensemble_id": k,
        "ensemble_name": v["ensemble_name"],
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"],
        "train_auc": v["train_auc"],
        "train_ks": v["train_ks"],
        "train_logloss": v["train_logloss"]
    }
    for k, v in ensemble_registry.items()
])

ensemble_summary = ensemble_summary.sort_values(
    by=["val_auc", "val_ks"],
    ascending=[False, False]
).reset_index(drop=True)

display(ensemble_summary)

,ensemble_id,ensemble_name,val_auc,val_ks,val_logloss,train_auc,train_ks,train_logloss
0,BAYSTACK002,Sparse Bayesian Ensemble,0.868719,0.587641,0.176104,0.872359,0.583831,0.174138
1,BAYSTACK004,Top 2 Bayesian Ensemble,0.868692,0.587523,0.175925,0.873310,0.584433,0.173337
2,WASTACK003,Weighted Average Ensemble (Multi-Start LogLoss...,0.868535,0.586129,0.175963,0.874407,0.588876,0.173224
3,WASTACK005,Reduced Sparse Weighted Average Ensemble,0.868522,0.585835,0.175971,0.874412,0.589123,0.173224
4,WASTACK001,Optimized Weighted Average Ensemble,0.868521,0.585882,0.175971,0.874414,0.589112,0.173224
5,WASTACK004,Weighted Average Ensemble (Multi-Start Brier O...,0.868398,0.587941,0.176110,0.871197,0.580839,0.173763
6,BAYSTACK003,Hybrid Bayesian Ensemble,0.867910,0.586270,0.177182,0.869198,0.577879,0.176688
7,BAYSTACK001,Bayesian / Performance Weighted Ensemble,0.867887,0.586589,0.177222,0.869141,0.577768,0.176747
8,XGBSTACK004,XGB Conservative Ensemble,0.867832,0.583066,0.176419,0.876359,0.591053,0.171599
9,XGBSTACK001,XGBoost Meta-Learner Ensemble,0.867825,0.584639,0.176503,0.876331,0.590809,0.171409


In [65]:
# ============================================================
# PHASE A - SAVE ALL MODEL ARTIFACTS
# Base Models + ALL Ensemble Models + Registries + Reports
# Uses existing project paths
# ============================================================

BASE_MODEL_DIR = MODEL_DIR / "base_models"
ENSEMBLE_MODEL_DIR = MODEL_DIR / "ensemble_models"

BASE_CONFIG_DIR = CONFIG_DIR / "base_models"
ENSEMBLE_CONFIG_DIR = CONFIG_DIR / "ensemble_models"

REPORT_DIR = OUTPUT_DIR / "reports"

for d in [
    BASE_MODEL_DIR,
    ENSEMBLE_MODEL_DIR,
    BASE_CONFIG_DIR,
    ENSEMBLE_CONFIG_DIR,
    REPORT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

# ============================================================
# 1. SAVE BASE MODELS
# ============================================================

base_model_rows = []

for model_id, model_obj in champion_models.items():

    model_path = BASE_MODEL_DIR / f"{model_id}.joblib"
    config_path = BASE_CONFIG_DIR / f"{model_id}.json"

    joblib.dump(model_obj, model_path)

    config = {
        "model_id": model_id,
        "artifact_type": "base_model",
        "saved_ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_path": str(model_path)
    }

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

    base_model_rows.append({
        "model_id": model_id,
        "model_path": str(model_path),
        "config_path": str(config_path)
    })

print(f"Base models saved: {len(base_model_rows)}")

# ============================================================
# 2. SAVE ALL ENSEMBLE MODELS AS JOBLIB PAYLOADS + CONFIGS + WEIGHTS
# This saves every ensemble, including formula/weight-only ensembles.
# ============================================================

ensemble_rows = []

for ensemble_id, meta in ensemble_registry.items():

    payload_path = ENSEMBLE_MODEL_DIR / f"{ensemble_id}.joblib"
    config_path = ENSEMBLE_CONFIG_DIR / f"{ensemble_id}.json"
    weights_path = ENSEMBLE_CONFIG_DIR / f"{ensemble_id}_weights.csv"

    weights_table = meta.get("weights_table", None)

    # Save weights table separately if available
    if weights_table is not None:
        try:
            weights_table.to_csv(weights_path, index=False)
            saved_weights_path = str(weights_path)
        except Exception as e:
            print(f"Warning: could not save weights table for {ensemble_id}: {e}")
            saved_weights_path = None
    else:
        saved_weights_path = None

    # Joblib payload for ALL ensembles
    payload = {
        "artifact_type": "ensemble_model",
        "ensemble_id": ensemble_id,
        "ensemble_name": meta["ensemble_name"],
        "ensemble_type": meta["ensemble_type"],
        "base_models": meta["base_models"],
        "model_object": meta.get("model_object", None),
        "weights_table": weights_table,
        "train_predictions": meta.get("train_predictions", None),
        "val_predictions": meta.get("val_predictions", None),
        "metrics": {
            "train_auc": float(meta["train_auc"]),
            "val_auc": float(meta["val_auc"]),
            "train_ks": float(meta["train_ks"]),
            "val_ks": float(meta["val_ks"]),
            "train_logloss": float(meta["train_logloss"]),
            "val_logloss": float(meta["val_logloss"])
        },
        "created_ts": meta.get("created_ts", None),
        "saved_ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "notes": meta.get("notes", None),
        "weights_path": saved_weights_path
    }

    joblib.dump(payload, payload_path)

    # JSON config
    config = {
        "artifact_type": "ensemble_model",
        "ensemble_id": ensemble_id,
        "ensemble_name": meta["ensemble_name"],
        "ensemble_type": meta["ensemble_type"],
        "base_models": meta["base_models"],
        "train_auc": float(meta["train_auc"]),
        "val_auc": float(meta["val_auc"]),
        "train_ks": float(meta["train_ks"]),
        "val_ks": float(meta["val_ks"]),
        "train_logloss": float(meta["train_logloss"]),
        "val_logloss": float(meta["val_logloss"]),
        "created_ts": meta.get("created_ts", None),
        "saved_ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "notes": meta.get("notes", None),
        "model_path": str(payload_path),
        "weights_path": saved_weights_path,
        "has_model_object": meta.get("model_object", None) is not None,
        "has_weights_table": weights_table is not None
    }

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

    ensemble_rows.append({
        "ensemble_id": ensemble_id,
        "ensemble_name": meta["ensemble_name"],
        "ensemble_type": meta["ensemble_type"],
        "val_auc": meta["val_auc"],
        "val_ks": meta["val_ks"],
        "val_logloss": meta["val_logloss"],
        "model_path": str(payload_path),
        "weights_path": saved_weights_path,
        "config_path": str(config_path),
        "has_model_object": meta.get("model_object", None) is not None,
        "has_weights_table": weights_table is not None
    })

print(f"Ensemble payloads/configs saved: {len(ensemble_rows)}")

# ============================================================
# 3. SAVE MASTER REGISTRIES
# ============================================================

joblib.dump(
    champion_models,
    REPORT_DIR / "base_model_registry.joblib"
)

joblib.dump(
    ensemble_registry,
    REPORT_DIR / "ensemble_registry.joblib"
)

print("Registries saved.")

# ============================================================
# 4. CREATE + SAVE TOURNAMENT TABLES
# ============================================================

base_model_manifest = pd.DataFrame(base_model_rows)
ensemble_manifest = pd.DataFrame(ensemble_rows)

final_ensemble_table = pd.DataFrame([
    {
        "ensemble_id": k,
        "ensemble_name": v["ensemble_name"],
        "ensemble_type": v["ensemble_type"],
        "base_models": ", ".join(v["base_models"]),
        "val_auc": v["val_auc"],
        "val_ks": v["val_ks"],
        "val_logloss": v["val_logloss"],
        "train_auc": v["train_auc"],
        "train_ks": v["train_ks"],
        "train_logloss": v["train_logloss"],
        "notes": v["notes"]
    }
    for k, v in ensemble_registry.items()
]).sort_values(
    by=["val_auc", "val_ks"],
    ascending=[False, False]
).reset_index(drop=True)

base_model_manifest.to_csv(
    REPORT_DIR / "base_model_manifest.csv",
    index=False
)

ensemble_manifest.to_csv(
    REPORT_DIR / "ensemble_manifest.csv",
    index=False
)

final_ensemble_table.to_csv(
    REPORT_DIR / "final_ensemble_leaderboard.csv",
    index=False
)

print("Tournament reports saved.")

# ============================================================
# 5. SAVE CHAMPION / CHALLENGER SUMMARY
# ============================================================

champion_summary = pd.DataFrame([
    {
        "role": "Champion Candidate",
        "model_id": "BAYSTACK004",
        "reason": "Best top-tier logloss with near-best AUC and simple two-model structure."
    },
    {
        "role": "AUC Leader",
        "model_id": "BAYSTACK002",
        "reason": "Highest validation AUC across the full ensemble tournament."
    },
    {
        "role": "Optimization Challenger",
        "model_id": "WASTACK003",
        "reason": "Best optimized weighted-average model."
    },
    {
        "role": "Nonlinear Challenger",
        "model_id": "XGBSTACK004",
        "reason": "Best XGBoost meta-learner."
    }
])

champion_summary.to_csv(
    REPORT_DIR / "champion_challenger_framework.csv",
    index=False
)

print("Champion framework saved.")

# ============================================================
# 6. VERIFY ALL ENSEMBLES SAVED
# ============================================================

saved_ensemble_files = sorted([p.stem for p in ENSEMBLE_MODEL_DIR.glob("*.joblib")])
registry_ensemble_ids = sorted(list(ensemble_registry.keys()))

missing_ensemble_files = sorted(set(registry_ensemble_ids) - set(saved_ensemble_files))
extra_ensemble_files = sorted(set(saved_ensemble_files) - set(registry_ensemble_ids))

print("Registry ensemble count:", len(registry_ensemble_ids))
print("Saved ensemble joblib count:", len(saved_ensemble_files))

if missing_ensemble_files:
    print("Missing ensemble joblib files:")
    print(missing_ensemble_files)
else:
    print("All registry ensembles saved as joblib payloads.")

if extra_ensemble_files:
    print("Extra joblib files in ensemble folder:")
    print(extra_ensemble_files)

# ============================================================
# 7. DISPLAY FINAL OUTPUTS
# ============================================================

display(final_ensemble_table)
display(champion_summary)
display(ensemble_manifest)

print("PHASE A COMPLETE.")
print("Reports saved to:", REPORT_DIR)
print("Base models saved to:", BASE_MODEL_DIR)
print("Ensemble models saved to:", ENSEMBLE_MODEL_DIR)
print("Configs saved to:", CONFIG_DIR)

Directories ready.
Base models saved: 7
Ensemble payloads/configs saved: 20
Registries saved.
Tournament reports saved.
Champion framework saved.
Registry ensemble count: 20
Saved ensemble joblib count: 20
All registry ensembles saved as joblib payloads.


,ensemble_id,ensemble_name,ensemble_type,base_models,val_auc,val_ks,val_logloss,train_auc,train_ks,train_logloss,notes
0,BAYSTACK002,Sparse Bayesian Ensemble,Inverse LogLoss Top Model Blend,"LGB002, CAT005, XGB004",0.868719,0.587641,0.176104,0.872359,0.583831,0.174138,Top model inverse-logloss Bayesian weighting.
1,BAYSTACK004,Top 2 Bayesian Ensemble,Inverse LogLoss Two Model Blend,"LGB002, CAT005",0.868692,0.587523,0.175925,0.873310,0.584433,0.173337,Two strongest model Bayesian blend.
2,WASTACK003,Weighted Average Ensemble (Multi-Start LogLoss...,Softmax Multi-Start LogLoss Weighted Average,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.868535,0.586129,0.175963,0.874407,0.588876,0.173224,Multiple random starts optimizing train log loss.
3,WASTACK005,Reduced Sparse Weighted Average Ensemble,Reduced Softmax LogLoss Weighted Average,"LGB002, CAT005",0.868522,0.585835,0.175971,0.874412,0.589123,0.173224,Reduced weighted ensemble keeping only models ...
4,WASTACK001,Optimized Weighted Average Ensemble,Softmax-Constrained Weighted Probability Average,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.868521,0.585882,0.175971,0.874414,0.589112,0.173224,Weights optimized on train log loss using posi...
5,WASTACK004,Weighted Average Ensemble (Multi-Start Brier O...,Softmax Multi-Start Brier Weighted Average,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.868398,0.587941,0.176110,0.871197,0.580839,0.173763,Multiple random starts optimizing train Brier ...
6,BAYSTACK003,Hybrid Bayesian Ensemble,AUC / LogLoss Evidence Weighted Blend,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.867910,0.586270,0.177182,0.869198,0.577879,0.176688,Weights proportional to AUC divided by logloss.
7,BAYSTACK001,Bayesian / Performance Weighted Ensemble,Inverse LogLoss Weighted Average,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.867887,0.586589,0.177222,0.869141,0.577768,0.176747,"Weights based on inverse train log loss, norma..."
8,XGBSTACK004,XGB Conservative Ensemble,XGBoost Blender,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.867832,0.583066,0.176419,0.876359,0.591053,0.171599,Low learning rate conservative XGB.
9,XGBSTACK001,XGBoost Meta-Learner Ensemble,XGBoost Blender,"LGB002, XGB004, CAT005, NN003, BNB018, WGNB012...",0.867825,0.584639,0.176503,0.876331,0.590809,0.171409,XGBoost meta-learner using base model probabil...


,role,model_id,reason
0,Champion Candidate,BAYSTACK004,Best top-tier logloss with near-best AUC and s...
1,AUC Leader,BAYSTACK002,Highest validation AUC across the full ensembl...
2,Optimization Challenger,WASTACK003,Best optimized weighted-average model.
3,Nonlinear Challenger,XGBSTACK004,Best XGBoost meta-learner.


,ensemble_id,ensemble_name,ensemble_type,val_auc,val_ks,val_logloss,model_path,weights_path,config_path,has_model_object,has_weights_table
0,EASTACK001,Equal Average Ensemble,Equal Probability Average,0.866949,0.585805,0.181768,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,False,True
1,WASTACK001,Optimized Weighted Average Ensemble,Softmax-Constrained Weighted Probability Average,0.868521,0.585882,0.175971,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
2,LRSTACK001,Logistic Regression Stacking Ensemble,Logistic Regression Meta-Learner,0.865814,0.586328,0.187264,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
3,RIDGESTACK001,Ridge Logistic Stacking Ensemble,L2-Regularized Logistic Meta-Learner,0.866623,0.587016,0.186805,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
4,LASSOSTACK001,Lasso Logistic Stacking Ensemble,L1-Regularized Logistic Meta-Learner,0.864675,0.585563,0.187735,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
5,RFSTACK001,Random Forest Meta-Learner Ensemble,Random Forest Blender,0.865967,0.584903,0.176657,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
6,XGBSTACK001,XGBoost Meta-Learner Ensemble,XGBoost Blender,0.867825,0.584639,0.176503,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True
7,RASTACK001,Rank Average Ensemble,Average of Model Prediction Ranks,0.866676,0.581106,0.823717,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,False,True
8,BAYSTACK001,Bayesian / Performance Weighted Ensemble,Inverse LogLoss Weighted Average,0.867887,0.586589,0.177222,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,False,True
9,NNSTACK001,Neural Network Blender Ensemble,MLP Meta-Learner,0.864773,0.586089,0.187964,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,/Users/sumanchattopadhyay/Documents/Documents ...,True,True


PHASE A COMPLETE.
Reports saved to: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/reports
Base models saved to: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/saved_models/base_models
Ensemble models saved to: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/saved_models/ensemble_models
Configs saved to: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/model_configs


# Phase B — Ensemble Tournament Summary

This notebook evaluated multiple ensemble approaches using the saved base model prediction matrices from the credit-risk modeling pipeline.

The goal was not to assume that one ensemble method would win, but to compare several families under a common validation framework.

## Models Evaluated

The ensemble tournament included:

- Equal average ensembles
- Optimized weighted average ensembles
- Logistic regression stacking
- Ridge and Lasso stacking
- Random forest meta-learners
- XGBoost meta-learners
- Rank averaging
- Bayesian / performance-weighted averaging
- Neural network blending

All models were evaluated using the same training and validation split.

## Key Result

The strongest models were sparse Bayesian and weighted-average blends.

The top validation performers were:

- `BAYSTACK002`
- `BAYSTACK004`
- `WASTACK003`
- `WASTACK005`
- `WASTACK001`

This shows that the best ensemble models did not require a highly complex second-stage learner. Simpler weighted blends performed better than random forest, neural network, and most XGBoost meta-learners.

## Main Modeling Insight

The strongest base models were already capturing most of the useful signal. Because of this, simple probability blending worked better than heavier nonlinear meta-models.

This is an important applied modeling result: more complex ensemble models do not always generalize better.

## Champion / Challenger View

Based on validation performance and practical deployment considerations:

- `BAYSTACK004` is the preferred champion candidate.
- `BAYSTACK002` is the AUC leader.
- `WASTACK003` is the strongest optimized weighted-average challenger.
- `XGBSTACK004` is the strongest nonlinear challenger.

## Interpretability

The leading models are interpretable at the ensemble level because they combine a small number of base model probability scores.

For example, `BAYSTACK004` uses a two-model Bayesian-weighted blend. This makes it easier to explain than a neural network or nonlinear stack:

- each base model produces a probability of default
- the ensemble combines those probabilities using documented weights
- the final output remains a probability score

This is useful in credit risk because the output can support:

- risk ranking
- approval strategy
- pricing segmentation
- manual review prioritization
- portfolio monitoring

## Why Ensembles Won

Ensembles performed well because different base models capture different patterns in the data. Combining them can reduce model-specific error and improve validation stability.

However, the tournament also showed that not all ensembles are useful. Some complex methods underperformed simpler approaches.

The final result supports a practical modeling principle:

> Use the simplest model that delivers strong validation performance and can be explained clearly.

## Artifact Status

All base models, ensemble models, configurations, weights, registries, and leaderboard outputs were saved to the project output directories.

These saved artifacts will be consumed in the next notebook for test-set scoring.

# Phase C — Champion / Challenger Governance Framework

The validation tournament produced several strong candidates. Rather than selecting only one model immediately, a champion / challenger framework is used.

This approach is standard in mature risk analytics environments because it balances performance, interpretability, and future monitoring.

## Recommended Champion

### `BAYSTACK004`

Reason:

- top-tier validation AUC
- strongest logloss among leading candidates
- only two base models
- easier to explain and maintain
- lower operational complexity

This makes it a strong production candidate.

## Primary Challenger

### `BAYSTACK002`

Reason:

- highest validation AUC in the tournament
- uses three elite base models
- excellent ranking performance

This model should be monitored alongside the champion.

## Optimization Challenger

### `WASTACK003`

Reason:

- strongest optimized weighted-average model
- confirms that probability blending is highly effective
- useful benchmark against Bayesian methods

## Nonlinear Challenger

### `XGBSTACK004`

Reason:

- best performing boosted meta-learner
- strongest nonlinear ensemble family
- useful benchmark if future data relationships become more nonlinear

## Monitoring Philosophy

No model should be assumed permanent.

After scoring the external Kaggle test set, rankings may change. Final champion selection should consider:

- out-of-sample AUC
- out-of-sample KS
- probability calibration
- stability versus validation results
- operational simplicity

## Current Decision Status

At this stage:

- `BAYSTACK004` = provisional champion
- `BAYSTACK002` = primary challenger
- `WASTACK003` = optimization challenger
- `XGBSTACK004` = nonlinear challenger

## Next Step

Use the saved artifacts in the next notebook to score the Kaggle holdout test set and confirm whether validation leadership holds on unseen data.

In [66]:
# ============================================================
# SAVE SCORING HANDOFF BUNDLE FOR 05_model_test.ipynb
# ============================================================

SCORING_HANDOFF_DIR = OUTPUT_DIR / "scoring_handoff"
SCORING_HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Save expected feature columns
# ------------------------------------------------------------

expected_columns = {
    "tree_columns": list(X_train_tree.columns),
    "scaled_columns": list(X_train_scaled.columns),
    "woe_columns": list(X_train_woe.columns),
    "bins_columns": list(X_train_bins.columns),
    "bins_ohe_columns": list(X_train_bins_ohe.columns),
}

with open(SCORING_HANDOFF_DIR / "expected_columns.json", "w") as f:
    json.dump(expected_columns, f, indent=4)

# ------------------------------------------------------------
# 2. Save base model feature map
# ------------------------------------------------------------

with open(SCORING_HANDOFF_DIR / "base_model_features.json", "w") as f:
    json.dump(features, f, indent=4)

# ------------------------------------------------------------
# 3. Save finalist model list
# ------------------------------------------------------------

finalist_models = {
    "production_candidate": "BAYSTACK004",
    "auc_leader": "BAYSTACK002",
    "weighted_average_challenger": "WASTACK003",
    "nonlinear_challenger": "XGBSTACK004",
    "single_model_benchmark": "LGB002",
    "required_base_models": ["LGB002", "CAT005", "XGB004"]
}

with open(SCORING_HANDOFF_DIR / "finalist_models.json", "w") as f:
    json.dump(finalist_models, f, indent=4)

# ------------------------------------------------------------
# 4. Save ensemble registry again for scoring
# ------------------------------------------------------------

joblib.dump(
    ensemble_registry,
    SCORING_HANDOFF_DIR / "ensemble_registry.joblib"
)

# ------------------------------------------------------------
# 5. Save model configs dictionary
# ------------------------------------------------------------

joblib.dump(
    model_configs,
    SCORING_HANDOFF_DIR / "model_configs.joblib"
)

# ------------------------------------------------------------
# 6. Create config file manifest
# ------------------------------------------------------------

config_manifest = []

for path in CONFIG_DIR.rglob("*"):
    if path.is_file():
        config_manifest.append({
            "file_name": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "full_path": str(path)
        })

config_manifest = pd.DataFrame(config_manifest)

config_manifest.to_csv(
    SCORING_HANDOFF_DIR / "config_manifest.csv",
    index=False
)

display(config_manifest)

print("Scoring handoff bundle saved to:", SCORING_HANDOFF_DIR)

,file_name,relative_path,full_path
0,LGB002_config.json,outputs/model_configs/LGB002_config.json,/Users/sumanchattopadhyay/Documents/Documents ...
1,CAT005_config.json,outputs/model_configs/CAT005_config.json,/Users/sumanchattopadhyay/Documents/Documents ...
2,GB007_config.json,outputs/model_configs/GB007_config.json,/Users/sumanchattopadhyay/Documents/Documents ...
3,binning_rules.xlsx,outputs/model_configs/binning_rules.xlsx,/Users/sumanchattopadhyay/Documents/Documents ...
4,RF002_config.json,outputs/model_configs/RF002_config.json,/Users/sumanchattopadhyay/Documents/Documents ...
...,...,...,...
59,BNB018.json,outputs/model_configs/base_models/BNB018.json,/Users/sumanchattopadhyay/Documents/Documents ...
60,NN003.json,outputs/model_configs/base_models/NN003.json,/Users/sumanchattopadhyay/Documents/Documents ...
61,WGNB012.json,outputs/model_configs/base_models/WGNB012.json,/Users/sumanchattopadhyay/Documents/Documents ...
62,CAT005.json,outputs/model_configs/base_models/CAT005.json,/Users/sumanchattopadhyay/Documents/Documents ...


Scoring handoff bundle saved to: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/scoring_handoff


In [67]:
# ============================================================
# SAVE SCORING HANDOFF README
# ============================================================

handoff_readme = """
# Scoring Handoff for 05_model_test.ipynb

## Purpose

This folder contains the metadata needed to score the Kaggle `cs-test.csv` file using the finalist models selected in `04e1_blending_voting_ensembles.ipynb`.

## Important Note

`cs-test.csv` does not contain true target labels. Therefore, `05_model_test.ipynb` is a production-style scoring notebook, not a model performance evaluation notebook.

## Files

- `expected_columns.json`
  - Expected feature columns for tree, scaled, WOE, binned, and one-hot binned matrices.

- `base_model_features.json`
  - Feature list expected by each saved base model.

- `finalist_models.json`
  - Finalist ensemble IDs and required base models.

- `ensemble_registry.joblib`
  - Full ensemble registry containing ensemble definitions, weights, metrics, and metadata.

- `model_configs.joblib`
  - Saved model configuration dictionary.

- `config_manifest.csv`
  - List of available config/mapping files under the project config directory.

## Finalist Models for Scoring

- BAYSTACK004: production candidate
- BAYSTACK002: AUC leader
- WASTACK003: weighted-average challenger
- XGBSTACK004: nonlinear challenger
- LGB002: single-model benchmark

## Required Base Models

- LGB002
- CAT005
- XGB004

## Notebook 05 Goal

Load raw `cs-test.csv`, recreate model-ready feature matrices, generate finalist model scores, assign risk bands, and export scored applicant output.

## No Refitting Rule

Notebook 05 should not train or fit any model. It should only load saved artifacts, apply saved transformations, and score.
"""

with open(SCORING_HANDOFF_DIR / "README.md", "w") as f:
    f.write(handoff_readme)

print("Scoring handoff README saved.")

Scoring handoff README saved.
